# TIES HPSC-ES Working Group, November 6, 2026: mixed precision computing with MPCR

Runtime > Change runtime type > R, Hardware accelerator T4 GPU before running. Without a GPU every cell still runs, and the GPU sections fall back to the CPU.

## Setup

MPCR stores a matrix in double, single, or half precision and runs the linear algebra on the CPU or on a GPU. It is compiled from source, so this cell takes a while (roughly 20 to 30 minutes). Run it once per session. It prints a progress line every 30 seconds.

In [ ]:
Sys.setenv(CUDA_HOME = "/usr/local/cuda", PATH = paste0("/usr/local/cuda/bin:", Sys.getenv("PATH")))  # so MPCR finds CUDA
log <- "mpcr_install.log"
cmd <- "Rscript -e 'install.packages(\"MPCR\", repos = \"https://cloud.r-project.org\")'"
system(paste0("(", cmd, " > ", log, " 2>&1; echo FINISHED >> ", log, ") &"))  # install in the background
repeat {  # print progress every 30 seconds
  Sys.sleep(30)
  out <- readLines(log, warn = FALSE)
  if ("FINISHED" %in% out) break
  out <- out[nzchar(trimws(out))]  # skip blank lines
  if (length(out)) cat(substr(trimws(tail(out, 1)), 1, 90), "\n")
  flush.console()
}
cat(if (requireNamespace("MPCR", quietly = TRUE)) "MPCR is installed.\n" else "Install failed: see mpcr_install.log\n")

## Computing environment

Which R, which BLAS, and which GPU this Colab machine has. The BLAS matters: R's reference BLAS is single-threaded and slow on large matrices, OpenBLAS and Intel MKL are multithreaded, and Apple's Accelerate is the fast one on a Mac.

In [ ]:
sessionInfo()  # R version, platform, BLAS and LAPACK libraries

In [ ]:
# GPU information (if any)
cat(system("nvidia-smi --query-gpu=name,memory.total --format=csv", intern = TRUE), sep = "\n")

In [ ]:
# CPU information
cat(system("lscpu | grep -E 'Model name|^CPU\\(s\\)'", intern = TRUE), sep = "\n")

In [ ]:
# System memory (RAM)
cat(system("free -h | head -2", intern = TRUE), sep = "\n")

## Store a covariance matrix in double and single precision

n random locations on the unit square, and the exponential covariance between them with effective range 0.1. `as.MPCR` makes a copy in the precision asked for: single precision takes half the memory of double. Everything from here to the GPU sections runs on the CPU, which is where MPCR places the linear algebra by default.

In [ ]:
library(MPCR)

In [ ]:
set.seed(1)
n <- 10240
x <- matrix(runif(2 * n), n)  # n random locations

In [ ]:
S <- exp(-2.9957 * as.matrix(dist(x)) / 0.1)  # exponential covariance, effective range 0.1

In [ ]:
S[1:3, 1:3]

In [ ]:
S64 <- as.MPCR(S, n, n, "double")  # 64-bit copy

In [ ]:
S32 <- as.MPCR(S, n, n, "single")  # 32-bit copy

In [ ]:
# Memory in GB
cat("R double:   ", round(as.numeric(object.size(S)) / 2^30, 3), "\n")
cat("MPCR double:", round(MPCR.object.size(S64) / 2^30, 3), "\n")
cat("MPCR single:", round(MPCR.object.size(S32) / 2^30, 3), "\n")

## CPU run, exponential covariance

Cholesky factorization on the CPU, three times: base R, MPCR in double, MPCR in single. Single is faster, and the factor it returns is close to the double one.

In [ ]:
system.time(L <- chol(S))  # Cholesky in base R (double precision)

In [ ]:
system.time(L64 <- chol(S64))  # Cholesky in double precision

In [ ]:
system.time(L32 <- chol(S32))  # Cholesky in single precision

In [ ]:
L64 <- MPCR.ToNumericMatrix(L64)  # back to an R matrix

In [ ]:
L32 <- MPCR.ToNumericMatrix(L32)

In [ ]:
max(abs(L64 - L32))  # largest entrywise difference

In [ ]:
# Log determinants
cat("double:", sprintf("%.6f", 2 * sum(log(diag(L64)))), "\n")
cat("single:", sprintf("%.6f", 2 * sum(log(diag(L32)))), "\n")

## CPU run, Matérn 5/2 covariance

Same locations and effective range, Matérn 5/2 covariance, still on the CPU. The matrix is closer to singular, and the rounding in single precision makes a pivot negative: Cholesky breaks down in low precision and the factorization stops with an error.

In [ ]:
rm(S, S64, S32, L, L64, L32); invisible(gc())  # free memory

In [ ]:
h <- 5.9154 * as.matrix(dist(x)) / 0.1  # scaled distances, effective range 0.1

In [ ]:
S <- (1 + h + h^2 / 3) * exp(-h); rm(h)  # Matern 5/2 covariance

In [ ]:
S64 <- as.MPCR(S, n, n, "double")  # 64-bit copy

In [ ]:
S32 <- as.MPCR(S, n, n, "single")  # 32-bit copy

In [ ]:
system.time(L64 <- chol(S64))  # Cholesky in double precision

In [ ]:
try(system.time(L32 <- chol(S32)))  # Cholesky in single precision: expect an error

## GPU run, exponential covariance

The same matrices, stored on the GPU, and the linear algebra placed there too.

In [ ]:
rm(S, S64, S32, L64); invisible(gc())  # free memory

In [ ]:
MPCR.SetOperationPlacement("default", "GPU")  # run the linear algebra on the GPU

In [ ]:
S <- exp(-2.9957 * as.matrix(dist(x)) / 0.1)  # same locations, exponential, effective range 0.1

In [ ]:
S64 <- as.MPCR(S, n, n, "double", "GPU")  # 64-bit copy on the GPU

In [ ]:
S32 <- as.MPCR(S, n, n, "single", "GPU")  # 32-bit copy on the GPU

In [ ]:
system.time(L64 <- chol(S64))  # Cholesky in double precision

In [ ]:
system.time(L32 <- chol(S32))  # Cholesky in single precision

In [ ]:
L64 <- MPCR.ToNumericMatrix(L64)

In [ ]:
L32 <- MPCR.ToNumericMatrix(L32)

In [ ]:
max(abs(L64 - L32))  # largest entrywise difference

In [ ]:
# Log determinants
cat("double:", sprintf("%.6f", 2 * sum(log(diag(L64)))), "\n")
cat("single:", sprintf("%.6f", 2 * sum(log(diag(L32)))), "\n")

## GPU run, Matérn 5/2 covariance

The breakdown in single precision happens on the GPU too.

In [ ]:
rm(S, S64, S32, L64, L32); invisible(gc())  # free memory

In [ ]:
h <- 5.9154 * as.matrix(dist(x)) / 0.1  # same locations, effective range 0.1

In [ ]:
S <- (1 + h + h^2 / 3) * exp(-h); rm(h)  # Matern 5/2 covariance

In [ ]:
S64 <- as.MPCR(S, n, n, "double", "GPU")  # 64-bit copy on the GPU

In [ ]:
S32 <- as.MPCR(S, n, n, "single", "GPU")  # 32-bit copy on the GPU

In [ ]:
system.time(L64 <- chol(S64))  # Cholesky in double precision

In [ ]:
try(system.time(L32 <- chol(S32)))  # Cholesky in single precision: expect an error

## Half precision on the GPU

16-bit storage is a quarter of double. On the GPU, MPCR offers the matrix product in half precision: `C16 = S16 %*% S16`. The entries come out rounded, so compare them with the product in double.

In [ ]:
rm(S, S64, S32, L64); invisible(gc())  # free memory

In [ ]:
S <- exp(-2.9957 * as.matrix(dist(x)) / 0.1)  # exponential covariance again

In [ ]:
S64 <- as.MPCR(S, n, n, "double", "GPU")  # 64-bit copy on the GPU

In [ ]:
S16 <- as.MPCR(S, n, n, "half", "GPU")  # 16-bit copy on the GPU

In [ ]:
C16 <- as.MPCR(matrix(0, n, n), n, n, "half", "GPU")  # 16-bit output

In [ ]:
# Memory in GB
cat("MPCR double:", round(MPCR.object.size(S64) / 2^30, 3), "\n")
cat("MPCR half:  ", round(MPCR.object.size(S16) / 2^30, 3), "\n")

In [ ]:
system.time(MPCR.gemm(S16, S16, C16))  # C16 = S16 %*% S16 in half precision

In [ ]:
P16 <- MPCR.ToNumericMatrix(C16)  # the half precision product, read back in double

In [ ]:
P64 <- S %*% S  # the same product in double, on the CPU

In [ ]:
rbind(double = P64[1, 1:3], half = P16[1, 1:3])  # the first three entries

In [ ]:
max(abs(P16 - P64) / P64)  # largest relative error

## Largest n on the GPU

Two n by n matrices and their product, in each precision. Halving the bytes per entry lets n grow by a factor of about 1.4. The values of n below fit a 16 GB T4; scale them with the memory of the GPU you were given.

In [ ]:
rm(S, S64, S16, C16, P16, P64); invisible(gc())  # free memory

In [ ]:
fit <- function(n, precision) {  # two n x n matrices on the GPU, and their product
  invisible(gc())  # free GPU memory from the previous call
  A <- new(MPCR, n * n, precision, "GPU"); A$ToMatrix(n, n)
  C <- new(MPCR, n * n, precision, "GPU"); C$ToMatrix(n, n)
  system.time(MPCR.gemm(A, A, C))  # C = A %*% A
}

In [ ]:
fit(25000, "double")  # 2 x 5.0 GB

In [ ]:
fit(35000, "single")  # 2 x 4.9 GB

In [ ]:
fit(50000, "half")  # 2 x 5.0 GB

## Mixed precision: a different precision per block

Back on the CPU. Split the matrix into nb by nb blocks, each m by m, and run the block Cholesky: the diagonal block is factorized in double, the blocks below it are solved for, and each is stored in the precision chosen by a rule. Here the rule is by position: the block just below the diagonal in double, everything farther away in single. The log determinant comes from the diagonal blocks, which stay in double.

In [ ]:
MPCR.SetOperationPlacement("default", "CPU")  # back to the CPU

In [ ]:
set.seed(1)
n <- 10240; nb <- 128; m <- n / nb  # nb x nb blocks, each m x m
x <- matrix(runif(2 * n), n)  # n random locations

In [ ]:
S <- exp(-2.9957 * as.matrix(dist(x)) / 0.1)  # exponential covariance, effective range 0.1

In [ ]:
I <- split(1:n, rep(1:nb, each = m))  # rows of each block

In [ ]:
L <- matrix(list(), nb, nb)  # Cholesky factor, stored block by block

In [ ]:
Lt <- matrix(0, n, n)  # the stored blocks, read back in double

In [ ]:
for (k in 1:nb) {
  d <- I[[k]]; before <- seq_len((k - 1) * m)  # this block, columns already done
  Lkk <- t(chol(S[d, d] - tcrossprod(Lt[d, before])))
  L[[k, k]] <- as.MPCR(Lkk, m, m, "double"); Lt[d, d] <- Lkk  # diagonal block: always double
  for (i in setdiff(k:nb, k)) {  # blocks below the diagonal
    r <- I[[i]]
    A <- S[r, d] - Lt[r, before] %*% t(Lt[d, before])
    A <- t(forwardsolve(Lkk, t(A)))
    p <- if (i - k == 1) "double" else "single"  # choose the precision of this block
    L[[i, k]] <- as.MPCR(A, m, m, p); Lt[r, d] <- MPCR.ToNumericMatrix(L[[i, k]])
  }
}

In [ ]:
2 * sum(log(diag(Lt)))  # log determinant from the mixed precision factor

In [ ]:
2 * sum(log(diag(chol(S))))  # the same in double, for comparison

In [ ]:
max(abs(Lt - t(chol(S))))  # largest entrywise difference between the two factors

Which blocks ended up in which precision: the picture of the factor. Only the diagonal and the first subdiagonal are in double.

In [ ]:
P <- matrix(NA_real_, nb, nb)
for (k in 1:nb) for (i in k:nb) P[i, k] <- if (i - k <= 1) 64 else 32  # the rule above

In [ ]:
cols <- c("32" = "#F26F22", "64" = "#4D4D4D")  # orange: single, dark grey: double
image(0:nb, 0:nb, t(ifelse(P == 64, 2, 1)[nb:1, ]), col = unname(cols), zlim = c(1, 2), asp = 1, axes = FALSE, xlab = "block column k", ylab = "block row i", main = sprintf("Per-block precision: fp32 %.1f%%, fp64 %.1f%% of the blocks below the diagonal", 100 * mean(P[lower.tri(P)] == 32), 100 * mean(P[lower.tri(P)] == 64)))
axis(1, at = seq(0, nb, by = 16)); axis(2, at = nb - seq(0, nb, by = 16), labels = seq(0, nb, by = 16), las = 1)
legend("topright", legend = c("fp32", "fp64"), fill = cols, border = NA, bty = "n")

## Same per-block rule, Matérn 5/2 covariance

The position rule does not know how ill-conditioned the matrix is. With the Matérn 5/2 covariance the single precision blocks lose too much, and the Cholesky of a later diagonal block fails. A rule that looks at the matrix, such as a per-block error budget, is the subject of the paper below.

In [ ]:
h <- 5.9154 * as.matrix(dist(x)) / 0.1  # scaled distances, effective range 0.1

In [ ]:
S <- (1 + h + h^2 / 3) * exp(-h); rm(h)  # Matern 5/2 covariance

In [ ]:
Lt <- matrix(0, n, n)  # start the factor again

In [ ]:
try(for (k in 1:nb) {
  d <- I[[k]]; before <- seq_len((k - 1) * m)
  Lkk <- t(chol(S[d, d] - tcrossprod(Lt[d, before])))  # expect an error at some block k
  L[[k, k]] <- as.MPCR(Lkk, m, m, "double"); Lt[d, d] <- Lkk
  for (i in setdiff(k:nb, k)) {
    r <- I[[i]]
    A <- S[r, d] - Lt[r, before] %*% t(Lt[d, before])
    A <- t(forwardsolve(Lkk, t(A)))
    p <- if (i - k == 1) "double" else "single"
    L[[i, k]] <- as.MPCR(A, m, m, p); Lt[r, d] <- MPCR.ToNumericMatrix(L[[i, k]])
  }
})

In [ ]:
k  # the block where it stopped